In [1]:
%load_ext autoreload
%autoreload 2


from index.inverted_index import Index
from index.documents import load_dataset, tokenizer
import pandas as pd

# msmarco = load_dataset("msmarco-passage/dev/judged")

enron = pd.read_csv("emails.csv")
test_docs = enron.iloc[:, 1:]

# print(enron.head(1))

# NUM_DOCS = 100000
# K = 10

In [2]:
# test_docs = enron.iloc[:NUM_DOCS]

print(test_docs[:5])

                                             message
0  Message-ID: <18782981.1075855378110.JavaMail.e...
1  Message-ID: <15464986.1075855378456.JavaMail.e...
2  Message-ID: <24216240.1075855687451.JavaMail.e...
3  Message-ID: <13505866.1075863688222.JavaMail.e...
4  Message-ID: <30922949.1075863688243.JavaMail.e...


In [3]:
from types import SimpleNamespace

class OrderedDataFrameDataset:
    def __init__(self, df, order, text_col="message"):
        self.df = df.reset_index(drop=True)
        self.order = list(order)
        self.text_col = text_col

    def docs_iter(self):
        for i in self.order:
            row = self.df.iloc[i]
            yield SimpleNamespace(
                text=row[self.text_col],
                doc_id=i
            )

    def docs_count(self):
        return len(self.df)

In [4]:
import random
from index.ordered_dataset import OrderedDataset

# docs = []
# for i, doc in enumerate(msmarco.docs_iter()):
#     docs.append(doc)
#     if i == NUM_DOCS - 1:
#         break

# order = list(range(1, NUM_DOCS + 1))
# random.shuffle(order)

random_dataset = OrderedDataFrameDataset(test_docs, range(len(test_docs)))
# random_dataset = OrderedDataFrameDataset(test_docs, range(NUM_DOCS))

unordered_index = Index()
unordered_index.build_index(random_dataset, tokenizer, max_num_docs=-1)

Processing:   99.9% [517000/517401] | Current docid: 517000              

In [5]:
print(unordered_index.average_gap_bit_width())

4.006567621738588


In [6]:
from index.reorder_utils import create_sparse_tf_idf_matrix
import pandas as pd

tf_idf_matrix, doc_ids, terms = create_sparse_tf_idf_matrix(unordered_index)
df = pd.DataFrame.sparse.from_spmatrix(tf_idf_matrix, index=doc_ids, columns=terms)


In [11]:
import warnings
from sklearn.exceptions import DataConversionWarning

warnings.filterwarnings("ignore", category=DataConversionWarning)

In [12]:
from time import perf_counter
import copy

def prepare_index(src_index, method):
    idx = copy.deepcopy(src_index)

    if(method == "none"):
        return idx.get_approx_size(), 0

    
    if not src_index.removed_tf : 
        idx.remove_tf()

    start = perf_counter()

    idx.compress_index(method)

    elapsed_time = perf_counter() - start
    
    return idx.get_approx_size(), elapsed_time


In [13]:
from index.reorder_utils import k_medoids, reorder_documents_by_tsp, tsp_path, calculate_distance_matrix
def TSP_order(df,k):

    print(f"using {k} total clusters")
    medoids, medoids_id, cluster_to_docs = k_medoids(df, k)
    
    dist_matrix = calculate_distance_matrix(medoids, method="jaccard")
    tsp_path_ = tsp_path(dist_matrix)
    new_order = reorder_documents_by_tsp(tsp_path_,cluster_to_docs, medoids_id)
    ordered_dataset = OrderedDataFrameDataset(test_docs, new_order)

    return ordered_dataset


In [14]:
print(df[:5])

   messageid  187829811075855378110javamailevansthyme  date       mon  \
1        0.0                                 1.720033   0.0  0.209325   
2        0.0                                        0   0.0         0   
3        0.0                                        0   0.0         0   
4        0.0                                        0   0.0  0.209325   
5        0.0                                        0   0.0         0   

         14       may      2001    163900      0700       pdt  ...  06cal  \
1  0.364028  0.171168  0.078636  1.157339  0.085284  0.085417  ...      0   
2         0  0.171168  0.078636         0  0.085284  0.085417  ...      0   
3         0         0         0         0  0.085284  0.085417  ...      0   
4         0         0         0         0  0.085284  0.085417  ...      0   
5         0         0         0         0  0.085284  0.085417  ...      0   

   37369  11cal  42057  258358611075842029959javamailevansthyme  \
1      0      0      0         

In [15]:
k_clusters = 10
ordered_dataset = TSP_order(df, k=k_clusters)

# 2. Build a new index using the TSP-ordered dataset
tsp_ordered_index = Index()
tsp_ordered_index.build_index(ordered_dataset, tokenizer, max_num_docs=-1)

# 3. Print the resulting LogGap (average gap bit width)
tsp_log_gap = tsp_ordered_index.average_gap_bit_width()
print(f"TSP Reordered LogGap: {tsp_log_gap:.4f}")

using 10 total clusters


c:\Users\Utente\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Impossibile trovare il file specificato
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Utente\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\Utente\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
  File "c:\Users\Utente\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\Ut

Processing:   99.9% [517000/517401] | Current docid: 517000              TSP Reordered LogGap: 3.5923
